# Cross-Mapping Evaluation Notebook

**Purpose:** Test both GPT-4o Mini and Gemini 2.5 Flash on 5 cross-mapping questions
that require the system to align ITGC controls with ISO 27001 Annex A controls.

**Output files:**
- `results_crossmap_gpt4o_raw.json`
- `results_crossmap_gemini_raw.json`
- `crossmap_responses_comparison.csv`

**Assumes:** `document_store_fixed.json`, `faiss_index.bin`, `bm25_corpus.pkl` are present.
Both `OPENAI_API_KEY` and `GEMINI_API_KEY` must be in your `.env` file.

## Cell 1 — Install dependencies

In [ ]:
!pip install openai google-generativeai langchain-google-genai ragas datasets langchain langchain-openai sentence-transformers faiss-cpu rank-bm25 pandas python-dotenv tqdm -q

## Cell 2 — Load environment

In [1]:
import os, json, time, pickle
import numpy as np
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

OPENAI_API_KEY = "Enter your key"
GEMINI_API_KEY = "Enter your key"
assert OPENAI_API_KEY, '❌ OPENAI_API_KEY not found'
assert GEMINI_API_KEY, '❌ GEMINI_API_KEY not found'
print('✅ Both API keys loaded')

DOCUMENT_STORE_PATH = 'document_store_fixed.json'
FAISS_INDEX_PATH    = 'faiss_index.bin'
BM25_PATH           = 'bm25_corpus.pkl'
EMBED_MODEL         = 'BAAI/bge-small-en-v1.5'
TOP_K               = 8
print(f'✅ Config ready | Embedding: {EMBED_MODEL} | Top-K: {TOP_K}')


✅ Both API keys loaded
✅ Config ready | Embedding: BAAI/bge-small-en-v1.5 | Top-K: 8


## Cell 3 — Load document store and indices

In [2]:
import faiss
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi

with open(DOCUMENT_STORE_PATH, 'r', encoding='utf-8') as f:
    document_store = json.load(f)

itgc_docs = [d for d in document_store if d['source'] == 'ITGC']
iso_docs  = [d for d in document_store if d['source'] == 'ISO27001']
print(f'✅ Document store loaded: {len(document_store)} total | ITGC={len(itgc_docs)} | ISO={len(iso_docs)}')

embed_model = SentenceTransformer(EMBED_MODEL)
print('✅ Embedding model loaded')

if os.path.exists(FAISS_INDEX_PATH):
    faiss_index = faiss.read_index(FAISS_INDEX_PATH)
    print(f'✅ FAISS index loaded: {faiss_index.ntotal} vectors')
else:
    print('⚠️  Rebuilding FAISS index...')
    texts = [d['raw_text'] for d in document_store]
    embeddings = embed_model.encode(texts, show_progress_bar=True, normalize_embeddings=True)
    faiss_index = faiss.IndexFlatIP(embeddings.shape[1])
    faiss_index.add(np.array(embeddings, dtype='float32'))
    faiss.write_index(faiss_index, FAISS_INDEX_PATH)
    print(f'✅ FAISS index built: {faiss_index.ntotal} vectors')

if os.path.exists(BM25_PATH):
    with open(BM25_PATH, 'rb') as f:
        bm25 = pickle.load(f)
    print('✅ BM25 index loaded')
else:
    print('⚠️  Rebuilding BM25...')
    corpus = [d['raw_text'].lower().split() for d in document_store]
    bm25 = BM25Okapi(corpus)
    with open(BM25_PATH, 'wb') as f:
        pickle.dump(bm25, f)
    print('✅ BM25 built')


✅ Document store loaded: 173 total | ITGC=37 | ISO=136


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

✅ Embedding model loaded
✅ FAISS index loaded: 158 vectors
✅ BM25 index loaded


## Cell 4 — Retrieval functions

In [3]:
def dense_search(query, top_k=10):
    qv = embed_model.encode(
        [f'Represent this sentence for searching relevant passages: {query}'],
        normalize_embeddings=True).astype('float32')
    scores, indices = faiss_index.search(qv, top_k)
    results = []
    for rank, (idx, score) in enumerate(zip(indices[0], scores[0])):
        if idx == -1: continue
        doc = document_store[idx].copy()
        doc['dense_score'] = float(score)
        results.append(doc)
    return results

def sparse_search(query, top_k=10):
    tokens = query.lower().split()
    scores = bm25.get_scores(tokens)
    top_indices = np.argsort(scores)[::-1][:top_k]
    results = []
    for rank, idx in enumerate(top_indices):
        if scores[idx] == 0: continue
        doc = document_store[idx].copy()
        doc['sparse_score'] = float(scores[idx])
        results.append(doc)
    return results

def hybrid_search(query, top_k=8, k_rrf=60):
    dense  = dense_search(query, top_k=top_k * 2)
    sparse = sparse_search(query, top_k=top_k * 2)
    rrf = {}
    for rank, doc in enumerate(dense):
        rrf[doc['doc_id']] = rrf.get(doc['doc_id'], 0) + 1/(k_rrf+rank+1)
    for rank, doc in enumerate(sparse):
        rrf[doc['doc_id']] = rrf.get(doc['doc_id'], 0) + 1/(k_rrf+rank+1)
    all_docs = {d['doc_id']: d for d in dense + sparse}
    ranked = sorted(rrf.items(), key=lambda x: x[1], reverse=True)[:top_k]
    return [all_docs[did] for did, _ in ranked]

def dual_source_search(query, k_each=4):
    """Force equal ITGC + ISO retrieval — essential for cross-mapping."""
    all_results = hybrid_search(query, top_k=30)
    itgc = [d for d in all_results if d['source'] == 'ITGC'][:k_each]
    iso  = [d for d in all_results if d['source'] == 'ISO27001'][:k_each]
    return itgc + iso

def format_context(docs):
    lines = []
    for i, doc in enumerate(docs):
        lines.append(f"[{i+1}] Source: {doc['source']} | ID: {doc['control_id']} | Domain: {doc['domain']}")
        lines.append(doc['raw_text'][:800])
        lines.append('')
    return '\n'.join(lines)

# Smoke test
test = dual_source_search('user access provisioning ISO 27001', k_each=2)
print('✅ Retrieval functions ready')
print(f'   Smoke test: {[d["control_id"] for d in test]}')


✅ Retrieval functions ready
   Smoke test: ['ITGC-ACC-07', 'ITGC-COM-05', 'ISO-7.2', 'ISO-4.1']


## Cell 5 — Cross-mapping system prompt

This uses the `compliance_check` prompt mode which returns a structured verdict.

In [4]:
CROSSMAP_SYSTEM_PROMPT = """You are a senior IT audit specialist performing cross-framework compliance mapping.
Your task is to evaluate whether the ITGC controls satisfy the requirements of the specified ISO 27001:2022 Annex A controls.
Use ONLY the provided context documents. Cite every finding with [N] notation.
Partial coverage is NOT full compliance — be precise.

Structure your response exactly as follows:
VERDICT: COVERED / PARTIALLY COVERED / NOT COVERED
REASONING: [explanation of why, citing [N] for each point]
COVERED BY: [which ITGC control IDs satisfy which ISO requirement]
GAPS: [what ISO requires that ITGC does not explicitly address]"""

print('✅ Cross-mapping system prompt defined')


✅ Cross-mapping system prompt defined


## Cell 6 — The 5 cross-mapping questions

Each question maps one or more ITGC controls to their ISO 27001 Annex A counterparts.

| Q | ITGC Controls | ISO Controls | Theme |
|---|---|---|---|
| CM1 | ACC-01, ACC-02, ACC-03 | ISO-5.15, 5.16, 5.18 | User lifecycle vs Access Control |
| CM2 | ACC-04 | ISO-8.2 | Privileged accounts vs Privileged access rights |
| CM3 | CHA-02, CHA-03 | ISO-5.3, 8.32 | Change SoD vs Segregation of duties |
| CM4 | COM-01, COM-04 | ISO-8.13, 8.15 | Backup + Monitoring vs Backup + Logging |
| CM5 | PRO-01, PRO-02, PRO-04 | ISO-8.25, 8.31 | Program dev vs Secure SDLC |

In [6]:
CROSSMAP_QUESTIONS = [
    {
        'q_index': 'CM1',
        'question': (
            'How should an organisation control who gets access to its systems, '
            'ensure that user identities are properly managed throughout their '
            'lifecycle, and make sure access rights are removed when no longer needed?'
        ),
        'itgc_ids': ['ITGC-ACC-01', 'ITGC-ACC-02', 'ITGC-ACC-03'],
        'iso_ids':  ['ISO-5.15', 'ISO-5.16', 'ISO-5.18'],
        'theme':    'User Lifecycle vs Access Control Framework',
    },
    {
        'q_index': 'CM2',
        'question': (
            'What controls should be in place to ensure that administrator and '
            'privileged access to systems is restricted only to those who need it, '
            'and how should the use of such elevated access be managed and reviewed?'
        ),
        'itgc_ids': ['ITGC-ACC-04'],
        'iso_ids':  ['ISO-8.2'],
        'theme':    'Privileged Access Controls Alignment',
    },
    {
        'q_index': 'CM3',
        'question': (
            'What controls should ensure that no single person has end-to-end control '
            'over a system change, and that all changes to IT systems go through '
            'a formal review and approval process before being implemented?'
        ),
        'itgc_ids': ['ITGC-CHA-02', 'ITGC-CHA-03'],
        'iso_ids':  ['ISO-5.3', 'ISO-8.32'],
        'theme':    'Change Management SoD and Approval vs ISO Controls',
    },
    {
        'q_index': 'CM4',
        'question': (
            'What controls should be in place to ensure critical data is regularly '
            'backed up and can be restored, and that system events, failures, '
            'and exceptions are recorded and retained for monitoring and review?'
        ),
        'itgc_ids': ['ITGC-COM-01', 'ITGC-COM-04'],
        'iso_ids':  ['ISO-8.13', 'ISO-8.15'],
        'theme':    'Backup and Monitoring vs ISO Backup and Logging',
    },
    {
        'q_index': 'CM5',
        'question': (
            'What controls should govern how software is developed securely, '
            'tested before release, and deployed in a way that keeps development, '
            'testing, and live production environments completely separate from each other?'
        ),
        'itgc_ids': ['ITGC-PRO-01', 'ITGC-PRO-02', 'ITGC-PRO-04'],
        'iso_ids':  ['ISO-8.25', 'ISO-8.31'],
        'theme':    'Program Development vs Secure SDLC and Environment Separation',
    },
]

print(f'✅ {len(CROSSMAP_QUESTIONS)} cross-mapping questions loaded')
for q in CROSSMAP_QUESTIONS:
    print(f"  {q['q_index']} | {q['theme']}")

✅ 5 cross-mapping questions loaded
  CM1 | User Lifecycle vs Access Control Framework
  CM2 | Privileged Access Controls Alignment
  CM3 | Change Management SoD and Approval vs ISO Controls
  CM4 | Backup and Monitoring vs ISO Backup and Logging
  CM5 | Program Development vs Secure SDLC and Environment Separation


## Cell 7 — Configure both model clients

In [7]:
from openai import OpenAI
import google.generativeai as genai

# GPT-4o Mini
gpt_client = OpenAI(api_key=OPENAI_API_KEY)
gpt_test = gpt_client.chat.completions.create(
    model='gpt-4o-mini', max_tokens=10,
    messages=[{'role':'user','content':'Reply only with: ready'}]
)
print(f"✅ GPT-4o Mini: '{gpt_test.choices[0].message.content.strip()}'")

# Gemini 2.5 Flash
genai.configure(api_key=GEMINI_API_KEY)
gem_client = genai.GenerativeModel('gemini-2.5-flash')
gem_test = gem_client.generate_content('Reply only with the word: ready')
print(f"✅ Gemini 2.5 Flash: '{gem_test.text.strip()}'")


/var/folders/vx/8r0lym995ss4yf12tdrmnknh0000gn/T/ipykernel_3073/3970658632.py:2: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


✅ GPT-4o Mini: 'ready'
✅ Gemini 2.5 Flash: 'ready'


## Cell 8 — RAG runner functions for both models

In [8]:
def run_crossmap_gpt(q_dict):
    """Run one cross-mapping question through GPT-4o Mini."""
    retrieved = dual_source_search(q_dict['question'], k_each=4)
    context   = format_context(retrieved)
    user_msg  = f'CONTEXT DOCUMENTS:\n{context}\n---\n\nQUESTION: {q_dict["question"]}'

    response = gpt_client.chat.completions.create(
        model='gpt-4o-mini',
        max_tokens=1500,
        temperature=0,
        messages=[
            {'role': 'system', 'content': CROSSMAP_SYSTEM_PROMPT},
            {'role': 'user',   'content': user_msg},
        ]
    )
    answer = response.choices[0].message.content

    return {
        'model':           'gpt-4o-mini',
        'q_index':         q_dict['q_index'],
        'theme':           q_dict['theme'],
        'question':        q_dict['question'],
        'itgc_ids':        q_dict['itgc_ids'],
        'iso_ids':         q_dict['iso_ids'],
        'answer':          answer,
        'context_ids':     [d['control_id'] for d in retrieved],
        'context_sources': [d['source'] for d in retrieved],
        'n_itgc_chunks':   sum(1 for d in retrieved if d['source'] == 'ITGC'),
        'n_iso_chunks':    sum(1 for d in retrieved if d['source'] == 'ISO27001'),
        'raw_context':     [d['raw_text'] for d in retrieved],
    }


def run_crossmap_gemini(q_dict):
    """Run one cross-mapping question through Gemini 2.5 Flash."""
    retrieved = dual_source_search(q_dict['question'], k_each=4)
    context   = format_context(retrieved)
    user_msg  = f'CONTEXT DOCUMENTS:\n{context}\n---\n\nQUESTION: {q_dict["question"]}'
    full_prompt = CROSSMAP_SYSTEM_PROMPT + '\n\n' + user_msg

    response = gem_client.generate_content(
        full_prompt,
        generation_config=genai.types.GenerationConfig(
            temperature=0,
            max_output_tokens=1500,
        )
    )
    answer = response.text

    return {
        'model':           'gemini-2.5-flash',
        'q_index':         q_dict['q_index'],
        'theme':           q_dict['theme'],
        'question':        q_dict['question'],
        'itgc_ids':        q_dict['itgc_ids'],
        'iso_ids':         q_dict['iso_ids'],
        'answer':          answer,
        'context_ids':     [d['control_id'] for d in retrieved],
        'context_sources': [d['source'] for d in retrieved],
        'n_itgc_chunks':   sum(1 for d in retrieved if d['source'] == 'ITGC'),
        'n_iso_chunks':    sum(1 for d in retrieved if d['source'] == 'ISO27001'),
        'raw_context':     [d['raw_text'] for d in retrieved],
    }


print('✅ RAG runner functions defined for both models')


✅ RAG runner functions defined for both models


## Cell 9 — Run all 5 questions through BOTH models

Runs GPT-4o Mini first then Gemini 2.5 Flash for each question.
Results saved to `results_crossmap_gpt4o_raw.json` and `results_crossmap_gemini_raw.json`.

In [9]:
gpt_results    = []
gemini_results = []

for i, q in enumerate(CROSSMAP_QUESTIONS):
    print(f'\n[{i+1}/5] {q["q_index"]} — {q["theme"]}')

    # GPT-4o Mini
    print('  Running GPT-4o Mini...')
    try:
        r_gpt = run_crossmap_gpt(q)
        gpt_results.append(r_gpt)
        print(f'  ✅ GPT | Chunks: ITGC={r_gpt["n_itgc_chunks"]} ISO={r_gpt["n_iso_chunks"]}')
        print(f'  Preview: {r_gpt["answer"][:150].strip()}...')
    except Exception as e:
        print(f'  ❌ GPT error: {e}')

    time.sleep(2)

    # Gemini 2.5 Flash
    print('  Running Gemini 2.5 Flash...')
    try:
        r_gem = run_crossmap_gemini(q)
        gemini_results.append(r_gem)
        print(f'  ✅ GEM | Chunks: ITGC={r_gem["n_itgc_chunks"]} ISO={r_gem["n_iso_chunks"]}')
        print(f'  Preview: {r_gem["answer"][:150].strip()}...')
    except Exception as e:
        print(f'  ❌ Gemini error: {e}')

    time.sleep(2)

# Save raw results
with open('results_crossmap_gpt4o_raw.json', 'w') as f:
    json.dump(gpt_results, f, indent=2, ensure_ascii=False)
with open('results_crossmap_gemini_raw.json', 'w') as f:
    json.dump(gemini_results, f, indent=2, ensure_ascii=False)

print(f'\n✅ All done. GPT results: {len(gpt_results)} | Gemini results: {len(gemini_results)}')
print('Saved: results_crossmap_gpt4o_raw.json')
print('Saved: results_crossmap_gemini_raw.json')



[1/5] CM1 — User Lifecycle vs Access Control Framework
  Running GPT-4o Mini...
  ✅ GPT | Chunks: ITGC=4 ISO=4
  Preview: VERDICT: PARTIALLY COVERED  
REASONING: The ITGC controls provide a framework for managing access to systems and ensuring user identities are properly...
  Running Gemini 2.5 Flash...
  ✅ GEM | Chunks: ITGC=4 ISO=4
  Preview: VERDICT: COVERED
REASONING: The ITGC controls collectively address the requirements for controlling system access, managing user identities throughout...

[2/5] CM2 — Privileged Access Controls Alignment
  Running GPT-4o Mini...
  ✅ GPT | Chunks: ITGC=4 ISO=4
  Preview: VERDICT: PARTIALLY COVERED  
REASONING: The ITGC controls provide some coverage for the management of privileged access. Specifically, ITGC-ACC-04 ens...
  Running Gemini 2.5 Flash...
  ✅ GEM | Chunks: ITGC=4 ISO=4
  Preview: VERDICT: COVERED
REASONING:
The ITGC controls collectively address the restriction, management, and review of administrator and privileged access to s...


## Cell 10 — Print full responses side by side

This is the key output to share back for qualitative analysis and thesis writing.

In [10]:
for i in range(len(CROSSMAP_QUESTIONS)):
    q = CROSSMAP_QUESTIONS[i]
    print('='*80)
    print(f"{q['q_index']} | {q['theme']}")
    print(f"ITGC: {q['itgc_ids']}  |  ISO: {q['iso_ids']}")
    print('='*80)
    print(f"QUESTION:\n{q['question']}")
    print()

    if i < len(gpt_results):
        r = gpt_results[i]
        print(f"--- GPT-4o Mini | Retrieved: {r['context_ids']} ---")
        print(r['answer'])
        print()

    if i < len(gemini_results):
        r = gemini_results[i]
        print(f"--- Gemini 2.5 Flash | Retrieved: {r['context_ids']} ---")
        print(r['answer'])
        print()
    print()


CM1 | User Lifecycle vs Access Control Framework
ITGC: ['ITGC-ACC-01', 'ITGC-ACC-02', 'ITGC-ACC-03']  |  ISO: ['ISO-5.15', 'ISO-5.16', 'ISO-5.18']
QUESTION:
How should an organisation control who gets access to its systems, ensure that user identities are properly managed throughout their lifecycle, and make sure access rights are removed when no longer needed?

--- GPT-4o Mini | Retrieved: ['ITGC-COM-05', 'ITGC-ACC-03', 'ITGC-ACC-06', 'ITGC-ACC-07', 'ISO-5.1', 'ISO-6.1.3', 'ISO-7.2', 'ISO-6.2'] ---
VERDICT: PARTIALLY COVERED  
REASONING: The ITGC controls provide a framework for managing access to systems and ensuring user identities are properly managed. Specifically, ITGC-COM-05 addresses the prevention of unauthorized access and changes to production systems, which aligns with the need for access control. ITGC-ACC-03 ensures timely removal of accounts for leavers or role changers, which is crucial for identity management. ITGC-ACC-06 and ITGC-ACC-07 further support password managem

## Cell 11 — Save comparison CSV

Flat CSV with both model responses per question for easy sharing.

In [10]:
rows = []
for i, q in enumerate(CROSSMAP_QUESTIONS):
    row = {
        'q_index': q['q_index'],
        'theme':   q['theme'],
        'itgc_ids': ' | '.join(q['itgc_ids']),
        'iso_ids':  ' | '.join(q['iso_ids']),
        'question': q['question'],
    }
    if i < len(gpt_results):
        row['gpt_answer']       = gpt_results[i]['answer']
        row['gpt_context_ids']  = ' | '.join(gpt_results[i]['context_ids'])
        row['gpt_n_itgc']       = gpt_results[i]['n_itgc_chunks']
        row['gpt_n_iso']        = gpt_results[i]['n_iso_chunks']
    if i < len(gemini_results):
        row['gemini_answer']      = gemini_results[i]['answer']
        row['gemini_context_ids'] = ' | '.join(gemini_results[i]['context_ids'])
        row['gemini_n_itgc']      = gemini_results[i]['n_itgc_chunks']
        row['gemini_n_iso']       = gemini_results[i]['n_iso_chunks']
    rows.append(row)

df = pd.DataFrame(rows)
df.to_csv('crossmap_responses_comparison.csv', index=False)
print('✅ Saved: crossmap_responses_comparison.csv')
print(f'Shape: {df.shape}')
print(df[['q_index','theme','gpt_n_itgc','gpt_n_iso','gemini_n_itgc','gemini_n_iso']].to_string(index=False))


✅ Saved: crossmap_responses_comparison.csv
Shape: (5, 13)
q_index                                                         theme  gpt_n_itgc  gpt_n_iso  gemini_n_itgc  gemini_n_iso
    CM1                    User Lifecycle vs Access Control Framework           4          4              4             4
    CM2                          Privileged Access Controls Alignment           4          4              4             4
    CM3            Change Management SoD and Approval vs ISO Controls           4          4              4             4
    CM4               Backup and Monitoring vs ISO Backup and Logging           4          4              4             4
    CM5 Program Development vs Secure SDLC and Environment Separation           4          4              4             4


In [ ]:
# ── Fill these AFTER reading responses ──────────────────────────────
# verdict_accuracy : 5 = correct verdict, 3 = partially correct, 1 = wrong
# gap_identification: 5 = all gaps identified, 1 = no gaps identified
# citation_quality  : 5 = cites both ITGC + ISO inline, 1 = no citations
# hallucination     : 5 = no fabrication, 1 = fabricated requirements
# usefulness        : 5 = audit-ready output, 1 = too vague to use

CROSSMAP_RUBRIC = [
    {'q_index':'CM1','model':'gpt-4o-mini',      'verdict_accuracy':None,'gap_identification':None,'citation_quality':None,'hallucination':None,'usefulness':None,'notes':''},
    {'q_index':'CM1','model':'gemini-2.5-flash',  'verdict_accuracy':None,'gap_identification':None,'citation_quality':None,'hallucination':None,'usefulness':None,'notes':''},
    {'q_index':'CM2','model':'gpt-4o-mini',      'verdict_accuracy':None,'gap_identification':None,'citation_quality':None,'hallucination':None,'usefulness':None,'notes':''},
    {'q_index':'CM2','model':'gemini-2.5-flash',  'verdict_accuracy':None,'gap_identification':None,'citation_quality':None,'hallucination':None,'usefulness':None,'notes':''},
    {'q_index':'CM3','model':'gpt-4o-mini',      'verdict_accuracy':None,'gap_identification':None,'citation_quality':None,'hallucination':None,'usefulness':None,'notes':''},
    {'q_index':'CM3','model':'gemini-2.5-flash',  'verdict_accuracy':None,'gap_identification':None,'citation_quality':None,'hallucination':None,'usefulness':None,'notes':''},
    {'q_index':'CM4','model':'gpt-4o-mini',      'verdict_accuracy':None,'gap_identification':None,'citation_quality':None,'hallucination':None,'usefulness':None,'notes':''},
    {'q_index':'CM4','model':'gemini-2.5-flash',  'verdict_accuracy':None,'gap_identification':None,'citation_quality':None,'hallucination':None,'usefulness':None,'notes':''},
    {'q_index':'CM5','model':'gpt-4o-mini',      'verdict_accuracy':None,'gap_identification':None,'citation_quality':None,'hallucination':None,'usefulness':None,'notes':''},
    {'q_index':'CM5','model':'gemini-2.5-flash',  'verdict_accuracy':None,'gap_identification':None,'citation_quality':None,'hallucination':None,'usefulness':None,'notes':''},
]

rubric_df = pd.DataFrame(CROSSMAP_RUBRIC)
print('Rubric template ready — fill scores above then re-run this cell')
print(rubric_df[['q_index','model','verdict_accuracy','gap_identification','citation_quality','hallucination','usefulness']].to_string(index=False))
